## 01 Document Loaders

In [ ]:
from langchain_community.document_loaders import PyPDFLoader, WebBaseLoader

# Load a website
url = "https://lilianweng.github.io/posts/2023-06-23-agent/"
loader = WebBaseLoader(url)
documents = loader.load()

print(f"Loaded {len(documents)} documents.")
print(documents[0].page_content[:200])


## 02 Text Splitters

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
chunks = text_splitter.split_documents(documents)
print(f"Split into {len(chunks)} chunks.")


## 03 Embeddings And Vectorstores

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# 1. Initialize the embedding model
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 2. Create the Vector Store from our chunks
vectorstore = Chroma.from_documents(
    documents=chunks, 
    embedding=embeddings, 
    persist_directory="./chroma_db"
)

# 3. Create a Retriever interface
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

results = retriever.invoke("What is Task Decomposition?")
print(f"Found {len(results)} relevant chunks.")


## 04 Sparse Retrieval (BM25)

In [ ]:
from langchain_community.retrievers import BM25Retriever

# Sparse retrieval relies on exact keyword matching (like a traditional search engine)
# It does NOT use neural embeddings.
bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 2

sparse_results = bm25_retriever.invoke("Task Decomposition")
print(f"Sparse retrieval found {len(sparse_results)} chunks based on exact keywords.")


## 05 Hybrid Retrieval (Ensemble)

In [ ]:
from langchain.retrievers import EnsembleRetriever

# Hybrid Retrieval combines the semantic understanding of Dense Vectors (Chroma)
# with the exact keyword matching of Sparse Vectors (BM25).
ensemble_retriever = EnsembleRetriever(
    retrievers=[retriever, bm25_retriever],
    weights=[0.5, 0.5] # Give equal weight to both search methods
)

hybrid_results = ensemble_retriever.invoke("How do agents use Task Decomposition?")
print(f"Hybrid retrieval found {len(hybrid_results)} highly accurate chunks.")
